# ఖర్చు వసూళ్ల విశ్లేషణ

ఈ నోట్‌బుక్‌లో స్థానిక రసీదు చిత్రాల నుంచి ప్రయాణ ఖర్చులని ప్రాసెస్ చేయడానికి ప్లగిన్లను ఉపయోగించే ఏజెంట్లను ఎలా సృష్టించాలో, ఖర్చు వసూళ్ల ఇమెయిల్‌ను సృష్టించుకోవటం, మరియు తుక్కు చార్ట్ ద్వారా ఖర్చుల డేటాను దృశ్యరూపంలో చూపించటం చూపిస్తారు. ఏజెంట్లు పనితీరు సందర్భాన్ని బట్టి ఫంక్షన్లను డైనమిక్‌గా ఎంచుకుంటాయి.

దశలు:
1. OCR ఏజెంట్ స్థానిక రసీదు చిత్రాన్ని ప్రాసెస్ చేసి ప్రయాణ ఖర్చుల డేటాను తీయ్యం.
2. ఇమెయిల్ ఏజెంట్ ఖర్చు వసూళ్ల ఇమెయిల్‌ను సృష్టిస్తారు.

### ప్రయాణ ఖర్చుల ఉదాహరణ:
మీరు మరో నగరంలో ఉన్న బిజినెస్ మీటింగ్ కోసం ప్రయాణిస్తున్న ఉద్యోగి అని ఊహించుకోండి. మీ కంపెనీ అన్ని సమంజసమైన ప్రయాణ సంబంధిత ఖర్చులను తిరిగి చెల్లించే విధానాన్ని కలిగి ఉంది. క్రింద ప్రయాణ ఖర్చుల వివరాలు ఉన్నాయి:
- రవాణా:
మీ స్వస్థల నగరం నుంచి గమ్యస్థల నగరానికి రౌండ్ ట్రిప్ విమాన టికెట్లు.
విమానాశ్రయానికి మరియు విమానాశ్రయంనుండి టాక్సీ లేదా రైడ్-హైలింగ్ సేవలు.
గమ్యస్థల నగరంలోని స్థానిక రవాణా (జనాభోస దేవల, అద్దె కార్లు, లేదా టాక్సీలు).

- నివాసం:
బిజినెస్ మీటింగ్ వేదిక దగ్గర మిడ్-రేంజ్ బిజినెస్ హోటల్‌లో మూడురోజుల హోటల్ స్థానం.

- భోజనం:
కంపెనీ నిత్యమయ్యే పా మెనూ విధానం ఆధారంగా ఉదయం, మధ్యాహ్నం, రాత్రి భోజనం అలవెన్స్.

- వివిధ ఖర్చులు:
విమానాశ్రయంలోని పార్కింగ్ ఫీజులు.
హోటల్‌లో ఇంటర్నెట్ యాక్సెస్ ఛార్జీలు.
చిట్లిపి లేదా చిన్న సేవా ఛార్జీలు.

- పత్రాల పూరకం:
మీరు అన్ని రసీదులు (ఫ్లైట్లు, టాక్సీలు, హోటల్, భోజనం మొదలగునవి) మరియు పూర్తిచేసిన ఖర్చు నివేదిక తిరిగి చెల్లింపుకు సమర్పిస్తారు.


## అవసరమైన లైబ్రరీలను దిగుమతి చేసుకోండి

నోట్‌బుక్ కోసం అవసరమైన లైబ్రరీలు మరియు మాడ్యూల్స్‌ను దిగుమతి చేసుకోండి.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## ఖర్చు నమూనాలను నిర్వచించండి

 వ్యక్తిగత ఖర్చుల కోసం Pydantic మోడల్‌ను సృష్టించి, యూజర్ ప్రశ్నను ర churrasqueira ఖర్చు డేటా రూపంలోకి మలచే ExpenseFormatter క్లాస్‌ను సృష్టించండి.

 ప్రతి ఖర్చు ఈ ఆకారంలో ప్రదర్శించబడుతుంది:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## సాధనాలను నిర్వచించడం - ఇమెయిల్ రూపొందించడం

ఖర్చుల వాదన సమర్పించేందుకు ఇమెయిల్ రూపొందించే సాధన ఫంక్షన్‌ను సృష్టించండి.
- ఈ సాధనం Microsoft ఏజెంట్ ఫ్రేమ్‌వర్క్ నుండి `@tool` డెకరేటర్‌ను ఉపయోగిస్తోంది.
- ఇది ఖర్చుల మొత్తం మొత్తం లెక్కించి, వివరాలను ఇమెయిల్ శరీరంగా ఫార్మాట్ చేస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను తీసుకునే సాధనం

రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను తీసుకునే సాధన కార్యాన్ని సృష్టించండి.
- ఈ సాధనం Microsoft Agent Framework నుండి `@tool` డెకరేటర్ ఉపయోగిస్తుంది.
- ఇది రసీదు చిత్రాన్ని చదవగా, దాన్ని base64 గా సంకేతీకరించి, ఏజెంట్ విశ్లేషించేందుకు data URIని తిరిగి ఇస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## ఖర్చుల ప్రాసెసింగ్

ఏజెంట్లను నిర్వచించి, వాటిని `WorkflowBuilder` ఉపయోగించి వరసై ఆపరేట్ చేసే వర్క్‌ఫ్లోగా కుట్టండి.
- OCR ఏజెంట్ రసీదు చిత్రం నుండి నిర్మిత ఖర్చుల డేటాను `load_receipt_image` సాధనం ద్వారా తీసుకుంటుంది.
- ఈమెయిల్ ఏజెంట్ తీసుకున్న డేటాని తీసుకుని `generate_expense_email` సాధనం ద్వారా ప్రొఫెషనల్ ఖర్చు క్లెయిమ్ ఇమెయిల్‌ను తయారు చేస్తుంది.
- `add_edge` తో `WorkflowBuilder` వరుస పైప్‌లైన్‌ను సృష్టిస్తుంది: OCR ఏజెంట్ → ఈమెయిల్ ఏజెంట్.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## ప్రధాన ఫంక్షన్

రసీదు చిత్రం ప్రాసెస్ చేయడానికి మరియు ఖర్చు క్లెయిమ్ ఇమెయిల్‌ని తయారు చేయడానికి వరుసగా వర్క్‌ఫ్లోను నిర్మించి, ఆపై దాన్ని నడిపి పూర్తి చేయండి.


> **గమనిక:** ఈ వర్క్‌ఫ్లో ప్రస్తుతం రసీదు చిత్రాన్ని base64 టెక్స్ట్‌గా అందజేస్తుంది, దీన్ని చాలా చాట్ మోడళ్లు (gpt-5-mini సహా) చిత్రం గా పరిగణించరు.
> ఇది మోడల్ కాంటెక్స్ట్ విండోని కూడా మించవచ్చు. Azure AI Vision (లేదా మరొక OCR సాధనం) తో OCR నడపడం ఇష్టపడండి మరియు మాత్రమే సేకరించిన టెక్స్ట్‌ను పంపండి, లేదా చిత్రాన్ని `image_url` సందేశంగా పంపేలా పునర్నిర్మించండి.
> మీరు కేవలం కాంటెక్స్ట్ లోపాలను నివారించాలనుకుంటే, చిన్న రసీదు చిత్రం లేదా పెద్ద కాంటెక్స్ట్ విండో ఉన్న మోడల్‌ని ప్రయత్నించండి.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
